# 02 — Skill Matching Evaluation

> **Requires a free API key.** Set `GROQ_API_KEY` (free at console.groq.com) or
> `GOOGLE_API_KEY` (free tier at aistudio.google.com) before running.
> Works locally or on Google Colab.

## Overview

The skill-matching pipeline has two stages:

1. **Extraction** — an LLM reads a job description and returns a JSON list of raw skill
   phrases (e.g. `'proficiency in Python'`, `'agile workflows'`).
2. **ESCO normalisation** — each phrase is embedded with `BAAI/bge-small-en-v1.5` and
   matched to the nearest ESCO skill label by cosine similarity, subject to a minimum
   threshold.

The threshold controls the **precision / recall tradeoff**:
- **Low threshold** → more matches → higher recall but more false positives (lower precision).
- **High threshold** → fewer matches → higher precision but misses legitimate skills.

This notebook sweeps thresholds from 0.45 to 0.80, computes macro F1 against a gold set,
and identifies the chosen operating point (~0.62).

In [ ]:
# fmt: off
import os, sys
from pathlib import Path

_cwd = Path.cwd()
_root = next(
    (p for p in [_cwd] + list(_cwd.parents)
     if (p / 'requirements.txt').exists() and (p / 'src').is_dir()),
    None,
)
assert _root, f'Could not find project root from {_cwd}. Open the notebook from inside the project tree.'
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f'Project root: {_root}')
# fmt: on


## 1. Load the gold set

In [ ]:
import json
from pathlib import Path

gold_path = Path('data/fixtures/gold_skills.json')
gold = json.loads(gold_path.read_text(encoding='utf-8'))
print(f'Gold entries: {len(gold)}')
for entry in gold[:2]:
    print('\nDescription snippet:', entry['description'][:120], '...')
    print('ESCO skills:', entry['esco_skills'])

## 2. Build the ESCO matcher

In [ ]:
from src.data.esco_loader import EscoIndex
from src.skills.esco_matcher import EscoMatcher

index = EscoIndex.load()
matcher = EscoMatcher(index)
print(f'Index size: {len(index.labels)} skills')

## 3. Define the extraction function

Needs an active LLM key.

In [ ]:
from src.generation.llm_client import build_chat_model, simple_complete
from src.skills.extractor import extract_skill_phrases

chat_model = build_chat_model(temperature=0.0)

def extract_fn(text: str) -> list[str]:
    return extract_skill_phrases(text, lambda p: simple_complete(p, chat_model))

print('LLM ready.')

## 4. Threshold sweep

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.eval.component_eval import evaluate

thresholds = np.round(np.arange(0.45, 0.81, 0.05), 2).tolist()
f1_scores = []

for t in thresholds:
    result = evaluate(
        gold_path,
        extract_fn,
        lambda phrases, th=t: matcher.match(phrases, threshold=th),
    )
    f1_scores.append(result['macro_f1'])
    print(f'threshold={t:.2f}  macro_F1={result["macro_f1"]:.4f}')

chosen = 0.62
plt.figure(figsize=(8, 4))
plt.plot(thresholds, f1_scores, marker='o', label='Macro F1')
plt.axvline(chosen, color='red', linestyle='--', label=f'Chosen threshold ({chosen})')
plt.xlabel('Cosine similarity threshold')
plt.ylabel('Macro F1')
plt.title('Skill-matching F1 vs cosine threshold')
plt.legend()
plt.tight_layout()
plt.savefig('notebooks/threshold_sweep.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Best F1 {max(f1_scores):.4f} at threshold {thresholds[f1_scores.index(max(f1_scores))]}')